# Ligand-based screening: machine learning

## Aim
Due to larger available data sources, machine learning (ML) gained momentum in drug discovery and especially in ligand-based virtual screening. In this notebook, we will learn how to use different supervised ML algorithms to predict the activity of novel compounds against our target of interest (EGFR).

## For more details

https://github.com/ICOA-SBC/TeachOpenCADD/blob/master/talktorials/7_machine_learning/T7_machine_learning.ipynb


## Instructions
Replace XXX with the appropriate code

## Configuration

In [ ]:
# Imports
# 1. Standard library imports
from pathlib import Path
import sys
sys.path.append('../my_modules') # to tell where to find local modules
import warnings

# 2. Third-party library imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rdkit import Chem 
from rdkit.Chem import (
    Draw,
    MACCSkeys,
    PandasTools
)
from rdkit.DataStructs import ConvertToNumpyArray

PandasTools.RenderImagesInAllDataFrames(images=True) # to molecules as images in DataFrames
from rdkit.Chem.Draw import IPythonConsole # needed to show molecules
# from rdkit.Chem.Draw.MolDrawing import MolDrawing, DrawingOptions # only needed if modifying defaults

import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    r2_score,
    mean_squared_error
)
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import recall_score
from sklearn.model_selection import KFold
from sklearn.neural_network import MLPClassifier

# 3. Local application imports
import kernel_infos

In [ ]:
# Information about the kernel
kernel_infos.show_kernel_info()

In [ ]:
# Global variables
HERE = Path().resolve()
print(f'{HERE}')
ROOT = HERE.parent
print(f'{ROOT}')
DATA = ROOT / 'data'
print(f'{DATA}')

## Example of linear regression

### Creation of synthetic data

In [ ]:
# Random data generation
np.random.seed(0)
X = np.random.rand(100, 1) * 10
y = 2.5 * X + np.random.randn(100, 1) * 2

# scatter plot data
plt.XXX(X, y, s=10)
plt.xlabel('X')
plt.ylabel('y')
# Define title to 'Synthetic data'
plt.XXX(XXX)
plt.show()

### Training of the model

In [ ]:
# Instantiate a model of LinearRegression
model = XXX()

# Train the model
model.XXX(X, y)

# Print the model coefficients
print(f"Slope: {model.coef_[0][0]:.2f}")
print(f"Ordinate at the origin: {model.intercept_[0]:.2f}")

### Results visualisation

In [ ]:
# Predict y_pred corresponding to X
y_pred = model.XXX(X)

# scatter plot synthetic data
plt.XXX(X, y, s=10)

# scatter plot y_pred in red
plt.XXX(X, y_pred, color=XXX)

plt.xlabel('X')
plt.ylabel('y')
plt.title('Linear regression on synthetic data')
plt.show()

In [ ]:
# Use plot instead of scatter for y_pred to visualise the model

# scatter plot synthetic data
plt.scatter(X, y, s=10)

# plot y_pred in red
plt.XXX(X, y_pred, color=XXX)

plt.xlabel('X')
plt.ylabel('y')
plt.title('Linear regression on synthetic data')
plt.show()

### Model evaluation

In [ ]:
# Calculate r2_score (coefficient of determination) between y and y_pred
R2 = r2_score(y, XXX)

# Print R2
print(f"R2: {XXX:.2f}")

In [ ]:
# Calculate Mean Squared Error between y and y_pred
MSE = mean_squared_error(y, XXX)

# Print MSE
print(f"MSE: {XXX:.2f} squared unit ")

### Data preparation
We will work on EGFR (Epidermal growth factor receptor) kinase data for now.

But before starting, we will define a helper function to help us creating the data frame we will work with, containing the following additional columns:
* ROMol: our molecules as molecule objects (created from smiles-strings)
* bv: The MACCS fingerprint MACCS as an python-object
* np_bv : the numpy array rtepresentation of the fingerprint

In [ ]:
def create_mol(df):
    # Construct a molecule from a smiles string
    # Generate mol column (ROMol): returns a Mol object, None on failure.
    df[XXX] = df[XXX].apply(Chem.MolFromSmiles)
    # Create a column for storing the molecular fingerprint as fingerprint object
    df['bv'] = df['ROMol'].apply(
        # Apply the lambda function  for each molecule
        lambda x: MACCSkeys.GenMACCSKeys(x)
    )
    # Allocate np.array to hold fp bit-vector (np = numpy)
    df['np_bv'] = np.zeros((len(df), df['bv'][0].GetNumBits())).tolist()
    df['np_bv'] = df['np_bv'].apply(np.array)
    # Convert the object fingerprint to NumpyArray and store in np_bv
    df.apply(lambda x: ConvertToNumpyArray(x['bv'], x['np_bv']), axis=1)
    return df

### Load data

In [ ]:
# File to load
EGFR_compounds_lipinski_csv_path = DATA / "EGFR_compounds_lipinski.csv"

In [ ]:
# Load the previous file with ',' as delimiter and colum 0 as index
EGFR_df = pd.read_csv(EGFR_compounds_lipinski_csv_path, delimiter=XXX, index_col=XXX)

In [ ]:
# Look at the dataframe info
EGFR_df.XXX

In [ ]:
# Display the first 2 rows of the dataframe
EGFR_df.head(XXX)

In [ ]:
# Change dataframe index to 'molecule_chembl_id'
EGFR_df.XXX('molecule_chembl_id', inplace=True)

In [ ]:
# Check the index modification
EGFR_df.XXX

### Classify data
A we need to classify each compound as **active (1)** or **inactive (0)**, we will use the `$pIC_{50}$` value $(pIC_{50} = -log_{10}(IC_{50}))$. A common cutoff value to discretize pIC50 data is **6.3**, which we will use for our experiment.

*NB: there are several other suggestions for an activity cut-off ranging from an $pIC_{50}$ value of 5 to 7 in the literature or even to define an exclusion range when not to take data points.*

In [ ]:
# Drop unnecessary columns: units and IC50
EGFR_df.drop(XXX, axis=1, inplace=True)

In [ ]:
# Check
EGFR_df.head(2)

In [ ]:
# Create molecules from smiles and calculate their fingerprints with the appropriate helper
ML_df = XXX(EGFR_df)

In [ ]:
# Add a column 'active' filled with zeros
ML_df[XXX] = np.zeros(len(ML_df))

# Mark every molecule with an pIC50 > 6.3 as active (=1.0)
ML_df.loc[ML_df[ML_df['pIC50'] >= XXX].index, 'active'] = 1.0

# Get the number of active molecules
ML_df["active"].XXX

In [ ]:
# Check
ML_df.head(1)

### Correlation

Correlation analysis is an extensively used technique that identifies **interesting relationships in data**. These relationships help us realize the relevance of attributes with respect to the target class to be predicted. 

In [ ]:
# List the columns of the dataframe
ML_df.XXX

In [ ]:
# Define a dataframe with only numeric columns (drop non numeric columns)
corr_df = ML_df.XXX(['smiles', 'ro5_fulfilled', 'ROMol', 'bv', 'np_bv'], axis=1)

In [ ]:
# Check
corr_df.head(2)

In [ ]:
# Check types
corr_df.XXX

`pairplot()` method from seaborn is used for **visualizing relationships between multiple variables** in a dataset. By creating a grid of scatter plots it helps to identify how different features interact with each other to identify patterns, correlations and trends in data.

In [ ]:
# Display plaiplot between the features from corr_df
sns.set_theme(style="ticks") # Set a consistent theme for all Seaborn subplots
sns.XXX(corr_df, hue="active")

In [ ]:

# Compute the correlation matrix between the previous variables and round values to 2 decimals
corr_matrix = corr_df.corr().XXX(2)

In [ ]:
# Dimensions of the correlation matrix
corr_matrix.XXX

In [ ]:
# Display the matrix
XXX

In [ ]:
# Plot the correlation matrix as a heatmap
plt.figure(figsize = (10,8)) # set figure size
sns.heatmap(XXX, annot = True)

The correlation matrix shows that **$pIC_{50}$ and active are strongly correlated**, which is expected since activity was defined directly from a $pIC_{50}$ cutoff. 

A more moderate positive correlation is also observable between MW and HBA, reflecting the tendency of larger molecules to contain more acceptor sites.

Negative correlations appear between **LogP and HBA-HBD**, consistent with the general trend that more polar molecules tend to be less lipophilic. All other descriptor pairs show relatively weak correlations, implying that most features provide independent information, which is advantageous for predictive modeling. 

Overall, these observations suggest the descriptors can collectively offer meaningful insights for distinguishing active and inactive compounds, despite the presence of $pIC_{50}$.

### Machine Learning (ML)

In the following section, we will try 2 ML and DL approaches to classify our molecules. We will use:
* Random Forest (RF)
* Artificial Neural Networks (ANNs) 

Before starting, we need first to define `X` (features) and `y` (target) and instanciate a **cross validation method** to split the dataset into n consecutive folds. Within each fold, data is splitted into *train/test sets* both for X and y.

But, because most machine learning models (like those in `scikit-learn`) expect **numerical input**, they do not understand `strings` or `objects`. Therefore, converting those types to numeric values is essential.

In [ ]:
# Define features (X) and target (y)
X = ML_df.drop('active', axis=1)
y = ML_df[XXX]

In [ ]:
# Helper function
def crossvalidation(model_l, X, y, n_folds=5):
    cm_l = []  # Store confusion matrices for each fold
    labels = -1 * np.ones(len(y))  # Store predicted labels for each data-point
    
    # Shuffle the indices for k-fold cross-validation
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    
    # Loop over folds
    for train_idx, test_idx in kf.split(X):

        # TRAIN
        train_x = X.iloc[train_idx].bv.tolist() # Convert the bit-vector and the label to a list
        train_y = y.iloc[train_idx].tolist()

        model_l.XXX(train_x, train_y) # Fit the model
        
        # TEST
        test_x = X.iloc[test_idx].bv.tolist()
        test_y = y.iloc[test_idx].tolist()

        # Prediction
        prediction_labels = model_l.XXX(test_x) # Predict
        
        # Store predicted labels
        labels[test_idx] = prediction_labels
        
        # Confusion matrix for this fold
        cm = confusion_matrix(test_y, prediction_labels)
        cm_l.append(cm)

    # Get overall accuracy, sensitivity, specificity
    acc = accuracy_score(y, labels)
    sens = recall_score(y, labels)
    spec = (acc * len(y) - sens * sum(y)) / (len(y) - sum(y))
    
    return acc, sens, spec, cm_l

Of course we want to assess the quality of our models. We will focus on:

* Sensitivity
* Specificity
* Accuracy
* Confusion Matrix

For reasons of clarity and comprehensibility of our code, we build helper functions to exploit our results. 


In [ ]:
# Helper function to help on model quality assesment
def print_results(acc, sens, spec):
    # Show overall accuracy, sensitivity, specificity
    print(f'Accuracy: {XXX}\nSensitivity: {XXX}\nSpecificity: {XXX}\n')
    print('\n')

In [ ]:
# Helper function to plot confusion matrix
def plot_confusion_matrix(cm):
    """
    Plot a 2x2 confusion matrix using seaborn heatmap
    with TN, FP, FN, TP annotations inside each cell.
    """
    plt.figure(figsize=(5,4))
    
    # Plot heatmap WITHOUT annotations
    sns.XXX(cm, annot=XXX, cmap="Blues", cbar=False, square=True)
    
    tn, fp, fn, tp = cm.ravel()
    
    # Coordinates: heatmap cells are centered at (col+0.5, row+0.5)
    # Annotate each cell
    annotations = [("TN", tn), ("FP", fp), ("FN", fn), ("TP", tp)]
    coords = [(0,0), (0,1), (1,0), (1,1)]
    
    for (label, value), (i,j) in zip(annotations, coords):
        plt.text(j+0.5, i+0.5, f"{label}={value}", ha='center', va='center', fontsize=12, color="white" if cm[i,j]>cm.max()/2 else "black")
    
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Confusion Matrix (last fold)")
    plt.xticks([0.5, 1.5], ["Inactive", "Active"])
    plt.yticks([0.5, 1.5], ["Inactive", "Active"], rotation=90)
    plt.show()

### Random forest classifier

Now we will start with a random forest classifier. We will first set the parameters. Afterwards we will do the cross validation of our model and plot the results. 

In [ ]:
# Set model's hyperparameters for random Forest
param = {'max_features': 'sqrt',
         'n_estimators': 2000,
         'criterion': 'entropy',
         'min_samples_leaf': 1}
modelRf = RandomForestClassifier(**param)

# Do cross-validation procedure with 5 folds with the right helper function
n = XXX
results = XXX(modelRf, X, y, n)

In [ ]:
# Assess model's quality  with the appropriate helper function
XXX(results[0], results[1], results[2])

In [ ]:
# Plot the confusion matrix of the last fold
plot_confusion_matrix(results[3][XXX])

Our models shows good values for all measured values and, thus, seem to be predictive.

### Neural network classifier
The last approach we try here is a neural network model. We train an MLPClassifier (Multi-layer Perceptron classifier) with 3 layers, each with 5 neurons. You may notice early stopping is explicitely set to FALSE. As before, we do the crossvalidation procedure and plot the results. For more infor on MLP, see [sklearn MLPClassifier](http://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html).

In [ ]:
# Hide convergence warnings
warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [ ]:
# Instantiate model, default activation: relu
modelClf = MLPClassifier(solver='adam', 
                         alpha=1e-5, 
                         hidden_layer_sizes=(5, 3), 
                         random_state=1, early_stopping=False)

# Do cross-validation procedure with 5 folds
n = XXX
results = XXX(modelClf, X, y, n)

In [ ]:
# Assess model's quality  with the appropriate helper function
print_results(XXX)

In [ ]:
# Plot the confusion matriw of the last fold
plot_confusion_matrix(results[3][XXX])